# 🧙 YOLO Training Wizard

CLI wizard interaktif untuk training model YOLO dengan dataset dari Roboflow — dioptimalkan dengan best practices dan smart defaults.

## Features
- **Roboflow Integration** — Paste snippet langsung dari Roboflow untuk download dataset
- **Training Presets** — Quick Test, Balanced, Maximum Quality, Small Objects, Fine-Tune
- **Auto-Batch** — Otomatis mendeteksi batch size optimal berdasarkan GPU memory
- **YOLO26 Smart Defaults** — Hyperparameter yang sudah dioptimalkan per model size
- **Augmentation Presets** — None, Light, Medium, Heavy, atau Custom
- **Model Export** — ONNX, TensorRT, CoreML, TFLite, OpenVINO, NCNN, TorchScript
- **Hyperparameter Tuning** — Mode otomatis untuk mencari parameter optimal
- **Classification Wizard** — Training YOLO classification dengan `classify_wizard.py`
- **Auto Detection→Classification Convert** — Crop bbox otomatis via `utils/convert_to_classification.py`

## Supported Models
| Generation | Variants |
|---|---|
| YOLO26 (Latest) | yolo26n, yolo26s, yolo26m, yolo26l, yolo26x |
| YOLO12 | yolo12n, yolo12s, yolo12m, yolo12l, yolo12x |
| YOLO11 | yolo11n, yolo11s, yolo11m, yolo11l, yolo11x |
| YOLOv8 | yolov8n, yolov8s, yolov8m, yolov8l, yolov8x |

## Classification Models
| Generation | Variants |
|---|---|
| YOLO26 (Latest) | yolo26n-cls, yolo26s-cls, yolo26m-cls, yolo26l-cls, yolo26x-cls |
| YOLO11 | yolo11n-cls, yolo11s-cls, yolo11m-cls, yolo11l-cls, yolo11x-cls |
| YOLOv8 | yolov8n-cls, yolov8s-cls, yolov8m-cls, yolov8l-cls, yolov8x-cls |

> Catatan: YOLO12 tidak memiliki pretrained classification weights, jadi tidak tersedia untuk classification wizard.

---

⚠️ **Pastikan runtime menggunakan GPU:** Runtime → Change runtime type → T4 GPU

## 1. Setup Environment

Install `uv` package manager dan clone repository.

In [ ]:
# Install uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh
!echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc

import os
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"

In [ ]:
# Clone repository
!git clone https://github.com/superrexy/yolo-training-wizard.git
%cd yolo-training-wizard

## 2. Verifikasi GPU

Pastikan GPU tersedia untuk training yang lebih cepat.

In [ ]:
# Cek GPU yang tersedia
!nvidia-smi

## 3. Aktifkan TensorBoard Logging

Aktifkan TensorBoard agar metrics training (loss, mAP, dll.) tercatat secara otomatis.
Lihat [dokumentasi resmi](https://docs.ultralytics.com/integrations/tensorboard/#usage) untuk detail lebih lanjut.

In [ ]:
# Aktifkan TensorBoard logging untuk YOLO
!yolo settings tensorboard=True

## 4. Jalankan YOLO Training Wizard

Wizard akan memandu Anda melalui:
1. Download dataset dari Roboflow (paste snippet atau manual)
2. Konfigurasi training (pilih preset atau custom)
3. Pre-training health checks
4. Training model dengan TensorBoard monitoring
5. Validasi model
6. Training summary & rekomendasi
7. Export model (ONNX/TensorRT/etc.)
8. Zip hasil training

> **Catatan:** Gunakan `%%shell` agar interactive input (Rich prompts) berfungsi dengan benar di Colab.

In [ ]:
%%shell
MPLBACKEND=Agg uv run main.py

## 4b. (Alternatif) Jalankan Classification Training Wizard

Gunakan wizard ini untuk training **YOLO Classification** (`classify_wizard.py`). Berbeda dari detection, classification memprediksi kelas gambar secara keseluruhan dan memakai metrik **Top-1/Top-5 Accuracy**.

Wizard classification mendukung:
1. Download dataset dari Roboflow (paste snippet atau manual)
2. Validasi minimal 2 class
3. Auto-convert dataset detection ke classification format jika diperlukan
4. Pilih model YOLO26/YOLO11/YOLOv8 dengan suffix `-cls`
5. Training, validation, summary, export, dan zip hasil di `runs/classify/`

> **Catatan:** Jalankan cell ini sebagai pengganti wizard detection jika target Anda adalah classification.


In [ ]:
%%shell
MPLBACKEND=Agg uv run classify_wizard.py

## 4c. (Opsional) Classification Hyperparameter Tuning Mode

Gunakan mode ini untuk mencari hyperparameter optimal khusus classification.

> ⚠️ Mode ini membutuhkan waktu lama. Jalankan cell ini **sebagai pengganti** cell classification wizard di atas.


In [ ]:
%%shell
MPLBACKEND=Agg uv run classify_wizard.py --tune

## 4d. (Alternatif) Detection Hyperparameter Tuning Mode

Gunakan mode ini untuk mencari hyperparameter optimal secara otomatis menggunakan Ultralytics built-in evolution.

> ⚠️ Mode ini membutuhkan waktu yang lama. Jalankan cell ini **sebagai pengganti** cell di atas, bukan keduanya.

In [ ]:
%%shell
MPLBACKEND=Agg uv run main.py --tune

## 5. Download Hasil Training

Setelah training selesai, download file hasil training ke komputer lokal Anda.

In [ ]:
# Tampilkan file hasil training
import glob

# Cari file zip hasil
zip_files = glob.glob('runs/**/*_results.zip', recursive=True)
pt_files = glob.glob('runs/**/weights/best.pt', recursive=True)

print('=== File Zip Hasil ===')
for f in zip_files:
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f'  {f} ({size_mb:.1f} MB)')

print('\n=== Best Model Weights ===')
for f in pt_files:
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f'  {f} ({size_mb:.1f} MB)')

In [ ]:
# Download file ke komputer lokal (Google Colab)
from google.colab import files

# Download zip hasil (uncomment yang diinginkan)
if zip_files:
    print(f'Downloading: {zip_files[0]}')
    files.download(zip_files[0])
elif pt_files:
    print(f'Downloading: {pt_files[0]}')
    files.download(pt_files[0])

## 6. (Opsional) Simpan ke Google Drive

Simpan hasil training ke Google Drive agar tidak hilang saat runtime berakhir.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy hasil ke Google Drive
import shutil

drive_dest = '/content/drive/MyDrive/yolo-training-results'
os.makedirs(drive_dest, exist_ok=True)

# Copy zip files
for f in zip_files:
    dest = os.path.join(drive_dest, os.path.basename(f))
    shutil.copy2(f, dest)
    print(f'Copied: {f} -> {dest}')

# Copy best.pt
for f in pt_files:
    dest = os.path.join(drive_dest, os.path.basename(f))
    shutil.copy2(f, dest)
    print(f'Copied: {f} -> {dest}')

print(f'\n✅ Hasil disimpan di Google Drive: {drive_dest}')